# Job-Application Decision Agent

This notebook runs the repository agent, validates the prepared C17–C50 case set, and keeps evaluator labels hidden until predictions are saved. The agent recommends one of: **apply**, **research**, **request human help**, or **skip**. It never submits an application.

In [ ]:
# Run once if the project is not installed in this notebook kernel.
import sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)

## 1. Validate the agent-visible dataset

This reads only `case-inputs.md`. It does **not** open `hidden-labels.md`.

In [ ]:
from job_agent.cases import load_text_cases

cases_path = PROJECT_ROOT / 'data' / 'simulated-cases' / 'case-inputs.md'
cases = load_text_cases(cases_path)
print(f'Validated {len(cases)} agent-visible cases: {cases[0].case_id}-{cases[-1].case_id}')
[(case.case_id, case.title) for case in cases[:5]]

## 2. Exercise the deterministic policy offline

This example requires no API key or credits. It verifies that four passing checks produce `apply`.

In [ ]:
from job_agent.models import Assessment, Check, CheckStatus
from job_agent.policy import decide
from job_agent.render import render

passed = Check(CheckStatus.PASS, ('source: evidence shown',), 'Requirement is supported.')
offline_assessment = Assessment(
    mandatory_requirements=passed,
    project_work_evidence=passed,
    posting_recency=passed,
    posting_reliability=passed,
    matched_evidence=('resume: relevant experience',),
)
print(render(decide(offline_assessment)))

## 3. Check API configuration

The project automatically loads `OPENAI_API_KEY` from the git-ignored `.env`. This cell reports only whether a key exists; it never prints the secret.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env', override=False)
API_READY = bool(os.getenv('OPENAI_API_KEY'))
print('API key configured:', API_READY)

## 4. Run one simulated case

This makes one API request and uses only the selected case's visible résumé and job-post evidence. It does not read evaluator labels.

In [ ]:
from datetime import date
from job_agent.agent import JobApplicationAgent

if API_READY:
    selected = cases[0]
    decision = JobApplicationAgent().analyze_text(
        selected.resume, selected.job_post, date(2026, 9, 1), selected.case_id
    )
    print(render(decision))
else:
    print('Skipped: configure OPENAI_API_KEY first.')

## 5. Run or resume all C17–C50 predictions

Uncomment the final line when API credits are available. Results are appended after each case, and completed case IDs are skipped on later runs.

In [ ]:
predictions_path = PROJECT_ROOT / 'results' / 'simulated-predictions.jsonl'
batch_command = [
    sys.executable, '-m', 'job_agent.batch',
    '--cases', str(cases_path),
    '--output', str(predictions_path),
]
print('Ready to run:', ' '.join(batch_command))
# subprocess.run(batch_command, cwd=PROJECT_ROOT, check=True)

## 6. Reveal labels and evaluate

Run this only after every prediction is saved. It produces a confusion matrix and the repository's hypothetical decision costs.

In [ ]:
import json
from job_agent.evaluate import evaluate, load_labels

if predictions_path.exists():
    predictions = [json.loads(line) for line in predictions_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if len(predictions) == len(cases):
        labels_path = PROJECT_ROOT / 'data' / 'simulated-cases' / 'hidden-labels.md'
        report = evaluate(predictions, load_labels(labels_path))
        print(json.dumps(report, indent=2))
    else:
        print(f'Not evaluated: {len(predictions)}/{len(cases)} predictions are saved.')
else:
    print('Not evaluated: no prediction file exists yet.')

## Interpretation boundary

These fictional cases test policy consistency. They do not establish real-world hiring accuracy, fairness, interview likelihood, offer likelihood, or candidate-specific opportunity value.